## Notebook para revisar o conteudó do Dia 01 do Curso Ultilizando o ollama em vez da API da open IA

## Importando Bibliotecas

In [ ]:

# imports
import os
from dotenv import load_dotenv
from scraper import fetch_website_contents
from IPython.display import Markdown, display
import ollama

# If you get an error running this cell, then please head over to the troubleshooting notebook!

ImportError: cannot import name 'messages_for' from 'scraper' (c:\Users\campo\OneDrive\Documents\GitHub\Estudos-IA\llm_engineering\Week_01\scraper.py)

## Listando Modelos disponivel localmente

In [6]:
!ollama list

NAME              ID              SIZE      MODIFIED    
gemma3:4b         a2af6cc3eb7f    3.3 GB    7 days ago     
gpt-oss:latest    17052f91a42e    13 GB     5 weeks ago    
phi3:latest       4f2222927938    2.2 GB    5 weeks ago    
gemma3:270m       e7d36fb2c3b3    291 MB    5 weeks ago    


## Escolhendo o modelo

In [2]:
model = 'gemma3:4b'

## Montando Mensagens para requisição

In [3]:
menssage = f"olá {model} Esta é minha primeira mensagem para você!"

messages = [{
    "role":"user",
    "content":menssage
}]

messages

[{'role': 'user',
  'content': 'olá gemma3:4b Esta é minha primeira mensagem para você!'}]

# Requisitando modelo

In [22]:
response = ollama.chat(
    model=model,
    messages=messages
)

## Visualizando Resposta do modelo

In [26]:
response_dict = response.model_dump()
if 'message'in response_dict:
    if response_dict['message'] is not None and 'content' in response_dict['message']:
        print(response_dict.get('message').get('content'))

Olá! Que legal ter sua primeira mensagem comigo! 😊 É um prazer te conhecer. Como posso te ajudar hoje?



## Obtendo informações do website

In [30]:
website = fetch_website_contents("https://ge.globo.com/futebol/times/vasco/")

## Prompts

In [32]:

system_prompt = """
You are a snarky assistant that analyzes the contents of a website,
and provides a short, snarky, humorous summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown. write in portuguese of the brazil
"""

In [31]:
# Define our user prompt

user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.

"""

## Exemplo de prompts de usuários e sistemas

In [35]:
messages = [
    {"role": "system", "content": "VOcê é um assistente debochado"},
    {"role": "user", "content": "Quando é 2 + 2?"}
]

response = ollama.chat(
    model='gemma3:4b',
    messages=messages
)

response_dict = response.model_dump()
if 'message'in response_dict:
    if response_dict['message'] is not None and 'content' in response_dict['message']:
        print(response_dict.get('message').get('content'))

Ah, que pergunta delicada! Deixa eu ver... 2 + 2 é... *pausa dramática* ...4.

É, eu sei, é um clássico. Não me faça pensar demais, por favor. 😄


In [ ]:
# See how this function creates exactly the format above
def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website}
    ]

In [42]:
# And now: call the OpenAI API. You will get very familiar with this!

def summarize(url):
    website = fetch_website_contents(url)
    response = ollama.chat(
        model='gemma3:4b',
        messages=messages_for(website)
    )
    
    response_dict = response.model_dump()
    if 'message'in response_dict:
        if response_dict['message'] is not None and 'content' in response_dict['message']:
            return  response_dict.get('message').get('content')
 

In [44]:
# A function to display this nicely in the output, using markdown

def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

In [45]:
display_summary("https://ge.globo.com/futebol/times/vasco/")

Essa página tá bem focada em manter o vasco-vascoleiro neurastênico, né? Cheio de notícias, resultados, e análises mais ou menos profundas sobre o time. E pra completar, um questionário chato pra ver se você tá satisfeito com a seleção de notícias, porque aparentemente eles não sabem o que você quer ler. Aparentemente o time tá em um caos, com demissão de treinador e derrota para o Fluminense. Que divertido!

In [46]:
display_summary("https://ge.globo.com/futebol/times/vasco/noticia/2026/02/22/fernando-diniz-e-demitido-do-vasco.ghtml")

Mais um técnico vai pro buraco no Vasco. Pedrinho agradece o "carinho" e o "esforço" de Fernando Diniz, que fez 56 jogos e só conseguiu 17 vitórias. Bruno Lazaroni vai ser o "interino" até o Vasco encontrar outro cara pra fazer a mesma merda. Que beleza!